# Experiment 7: AutoGluon Tabular with categorical cleaning

This reruns the strong tabular AutoGluon ensemble after the shared `RentalPreprocessor` categorical cleaning update. Validation is chronological, and test categories remain strings rather than being restricted to a train-only category vocabulary.

In [2]:
# If needed: pip install autogluon.tabular
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'),
    Path('/content/drive/MyDrive/KDDM-project'),
]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

from autogluon.common import space
from autogluon.tabular import TabularPredictor
from preprocessing import RentalPreprocessor, load_datasets

RANDOM_STATE = 42
TARGET = 'MONTHLY_RENT'
TIME_LIMIT_SECONDS = int(os.environ.get('AUTOGLUON_TIME_LIMIT', '1800'))
HPO_TRIALS_PER_MODEL = 25
HPO_KWARGS = {'num_trials': HPO_TRIALS_PER_MODEL, 'scheduler': 'local', 'searcher': 'random'}
HPO_HYPERPARAMETERS = {
    'GBM': {
        'num_leaves': space.Int(16, 128),
        'num_boost_round': space.Int(200, 1500),
        'max_depth': space.Int(3, 12),
        'learning_rate': space.Real(0.01, 0.20, log=True),
        'min_data_in_leaf': space.Int(10, 150),
        'feature_fraction': space.Real(0.60, 1.00),
        'bagging_fraction': space.Real(0.60, 1.00),
    },
    'CAT': {
        'iterations': space.Int(200, 1500),
        'depth': space.Int(4, 10),
        'learning_rate': space.Real(0.01, 0.20, log=True),
        'l2_leaf_reg': space.Real(1.0, 30.0, log=True),
    },
}


In [3]:
data_dir, datasets = load_datasets(project_root / 'data')
raw_train = datasets['train'].copy()
raw_test = datasets['test'].copy()
raw_train['period'] = pd.to_datetime(raw_train['RENT_APPROVAL_DATE'] + '-01')
raw_test['period'] = pd.to_datetime(raw_test['RENT_APPROVAL_DATE'] + '-01')

# Same 80% chronological split convention as the earlier tabular AutoGluon experiment.
periods = np.sort(raw_train['period'].unique())
validation_cutoff = pd.Timestamp(periods[int(len(periods) * 0.80)])
raw_fit = raw_train[raw_train['period'] < validation_cutoff].drop(columns='period').copy()
raw_valid = raw_train[raw_train['period'] >= validation_cutoff].drop(columns='period').copy()

# The preprocessor prints every categorical field's before/after unique counts.
validation_preprocessor = RentalPreprocessor(datasets)
fit_df = validation_preprocessor.fit_transform(raw_fit)
valid_df = validation_preprocessor.transform(raw_valid)

final_preprocessor = RentalPreprocessor(datasets)
full_train_df = final_preprocessor.fit_transform(raw_train.drop(columns='period'))
test_df = final_preprocessor.transform(raw_test.drop(columns='period'))

print(f'Validation cutoff: {validation_cutoff:%Y-%m}')
print(f'Fit rows: {fit_df.shape}; validation rows: {valid_df.shape}')
print(f'Full train: {full_train_df.shape}; test: {test_df.shape}')


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


In [4]:
def prepare_features(frame):
    result = frame.drop(columns=[TARGET], errors='ignore').copy()
    result['RENT_APPROVAL_DATE'] = pd.to_datetime(result['RENT_APPROVAL_DATE'] + '-01')
    return result

X_fit = prepare_features(fit_df)
X_valid = prepare_features(valid_df)
X_full = prepare_features(full_train_df)
X_test = prepare_features(test_df)
y_fit = fit_df[TARGET].astype(float)
y_valid = valid_df[TARGET].astype(float)
y_full = full_train_df[TARGET].astype(float)

categorical_columns = X_fit.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
category_summary = pd.DataFrame([
    {
        'feature': column, 'fit_unique': X_fit[column].nunique(dropna=False),
        'validation_unique': X_valid[column].nunique(dropna=False),
        'test_unique': X_test[column].nunique(dropna=False),
        'unseen_test_values': len(set(X_test[column].dropna()) - set(X_full[column].dropna())),
    }
    for column in categorical_columns
]).sort_values('feature')
display(category_summary)
assert len(X_test) == len(raw_test)
print('Test categorical values are retained as strings; no train-only category vocabulary is imposed.')


,feature,fit_unique,validation_unique,test_unique,unseen_test_values
1,BLOCK,2699,2585,2660,11
4,FLAT_MODEL,19,20,19,0
3,FLAT_TYPE,6,6,6,0
5,REGION,5,5,5,0
2,STREET,574,573,573,0
0,TOWN,26,26,26,0


Test categorical values are retained as strings; no train-only category vocabulary is imposed.


In [5]:
validation_predictor = TabularPredictor(
    label=TARGET, problem_type='regression', eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_exp7_categorical_validation'),
).fit(
    train_data=X_fit.assign(**{TARGET: y_fit.to_numpy()}),
    tuning_data=X_valid.assign(**{TARGET: y_valid.to_numpy()}),
    hyperparameters=HPO_HYPERPARAMETERS, hyperparameter_tune_kwargs=HPO_KWARGS,
    presets='medium_quality', time_limit=TIME_LIMIT_SECONDS, verbosity=2,
)

valid_predictions = validation_predictor.predict(X_valid).astype(float)
validation_metrics = pd.DataFrame([{
    'RMSE': mean_squared_error(y_valid, valid_predictions) ** 0.5,
    'MAE': mean_absolute_error(y_valid, valid_predictions),
    'R2': r2_score(y_valid, valid_predictions),
}])
display(validation_metrics)
display(validation_predictor.leaderboard(X_valid.assign(**{TARGET: y_valid.to_numpy()}), silent=True, extra_info=True))


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.57 GB / 16.00 GB (22.3%)
Disk Space Avail:   194.22 GB / 460.38 GB (42.2%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp7_categorical_validation"
Train Data Rows:    117988
Train Data Columns: 31
Tuning Data Rows:    32012
Tuning Data Columns: 31
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feat

[1000]	valid_set's rmse: 473.216


 36%|███▌      | 9/25 [00:24<00:46,  2.90s/it]

[1000]	valid_set's rmse: 472.933


 40%|████      | 10/25 [00:31<01:03,  4.21s/it]

...

 72%|███████▏  | 18/25 [01:11<00:33,  4.82s/it]

[1000]	valid_set's rmse: 471.461


100%|██████████| 25/25 [01:42<00:00,  4.10s/it]
Fitted model: LightGBM/T1 ...
	-538.5148	 = Validation score   (-root_mean_squared_error)
	0.65s	 = Training   runtime
	0.02s	 = Validation runtime
Fitted model: LightGBM/T2 ...
	-473.52	 = Validation score   (-root_mean_squared_error)
	2.46s	 = Training   runtime
	0.08s	 = Validation runtime
Fitted model: LightGBM/T3 ...
	-475.6079	 = Validation score   (-root_mean_squared_error)
	1.78s	 = Training   runtime
	0.04s	 = Validation runtime
Fitted model: LightGBM/T4 ...
	-473.865	 = Validation score   (-root_mean_squared_error)
	3.75s	 = Training   runtime
	0.07s	 = Validation runtime
Fitted model: LightGBM/T5 ...
	-473.6929	 = Validation score   (-root_mean_squared_error)
	0.71s	 = Training   runtime
	0.03s	 = Validation runtime
Fitted model: LightGBM/T6 ...
	-474.1384	 = Validation score   (-root_mean_squared_error)
	2.94s	 = Training   runtime
	0.06s	 = Validation runtime
Fitted model: LightGBM/T7 ...
	-473.05	 = Validation score   (-root

,RMSE,MAE,R2
0,466.432303,345.73183,0.49597


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,...,hyperparameters,hyperparameters_fit,ag_args_fit,features,compile_time,child_hyperparameters,child_hyperparameters_fit,child_ag_args_fit,ancestors,descendants
0,WeightedEnsemble_L2,-466.432303,-466.432303,root_mean_squared_error,0.919113,0.751830,168.109143,0.001571,0.000329,0.027096,...,"{'use_orig_features': False, 'valid_stacker': ...",{},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[LightGBM/T7, CatBoost/T8, CatBoost/T19, CatBo...",None,"{'ensemble_size': 25, 'subsample_size': 1000000}",{'ensemble_size': 14},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[LightGBM/T7, CatBoost/T8, CatBoost/T19, CatBo...",[]
1,CatBoost/T8,-470.110903,-470.110903,root_mean_squared_error,0.044971,0.049467,30.525833,0.044971,0.049467,30.525833,...,"{'iterations': 859, 'learning_rate': 0.0866093...",{'iterations': 816},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[WeightedEnsemble_L2]
2,CatBoost/T19,-470.349685,-470.349685,root_mean_squared_error,0.059685,0.059240,66.919558,0.059685,0.059240,66.919558,...,"{'iterations': 1367, 'learning_rate': 0.064744...",{'iterations': 1004},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[WeightedEnsemble_L2]
3,CatBoost/T4,-470.405895,-470.405895,root_mean_squared_error,0.046911,0.039414,32.963205,0.046911,0.039414,32.963205,...,"{'iterations': 800, 'learning_rate': 0.0418269...",{'iterations': 799},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[]
4,CatBoost/T9,-470.669978,-470.669978,root_mean_squared_error,0.047169,0.051203,26.753069,0.047169,0.051203,26.753069,...,"{'iterations': 1489, 'learning_rate': 0.097046...",{'iterations': 1488},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[WeightedEnsemble_L2]
5,CatBoost/T23,-470.754395,-470.754395,root_mean_squared_error,0.037438,0.041544,20.908365,0.037438,0.041544,20.908365,...,"{'iterations': 568, 'learning_rate': 0.0915807...",{'iterations': 564},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[]
6,CatBoost/T15,-470.781847,-470.781847,root_mean_squared_error,0.047518,0.052167,34.734976,0.047518,0.052167,34.734976,...,"{'iterations': 1343, 'learning_rate': 0.029239...",{'iterations': 1343},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[WeightedEnsemble_L2]
7,CatBoost/T18,-471.144351,-471.144351,root_mean_squared_error,0.054839,0.057518,39.101503,0.054839,0.057518,39.101503,...,"{'iterations': 1355, 'learning_rate': 0.057148...",{'iterations': 1351},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[]
8,CatBoost/T2,-471.166742,-471.166742,root_mean_squared_error,0.036063,0.033337,17.040858,0.036063,0.033337,17.040858,...,"{'iterations': 759, 'learning_rate': 0.0608433...",{'iterations': 754},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[]
9,LightGBM/T19,-471.332182,-471.332182,root_mean_squared_error,0.207763,0.142733,2.506999,0.207763,0.142733,2.506999,...,"{'learning_rate': 0.09047101764422108, 'num_le...",{'num_boost_round': 1188},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[WeightedEnsemble_L2]


In [6]:
final_predictor = TabularPredictor(
    label=TARGET, problem_type='regression', eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_exp7_categorical_final'),
).fit(
    train_data=X_full.assign(**{TARGET: y_full.to_numpy()}),
    hyperparameters=HPO_HYPERPARAMETERS, hyperparameter_tune_kwargs=HPO_KWARGS,
    presets='medium_quality', time_limit=TIME_LIMIT_SECONDS, verbosity=2,
)

test_predictions = final_predictor.predict(X_test).astype(float).clip(lower=0)
diagnostic_output = test_df.copy()
diagnostic_output['PREDICTED_MONTHLY_RENT'] = test_predictions.to_numpy()
diagnostic_path = experiment_dir / 'exp_7_autogluon_categorical_clean_predictions.csv'
diagnostic_output.to_csv(diagnostic_path, index=False)

competition_submission = pd.DataFrame({'Id': raw_test.index, 'Predicted': test_predictions.to_numpy()})
assert competition_submission['Id'].is_unique and competition_submission['Predicted'].notna().all()
submission_path = experiment_dir / 'exp_7_autogluon_categorical_clean_submission.csv'
competition_submission.to_csv(submission_path, index=False)
print(f'Saved diagnostics to {diagnostic_path}')
print(f'Saved {len(competition_submission):,} competition predictions to {submission_path}')
display(competition_submission.head())


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.35 GB / 16.00 GB (21.0%)
Disk Space Avail:   194.02 GB / 460.38 GB (42.1%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp7_categorical_final"
Train Data Rows:    150000
Train Data Columns: 31
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting AutoML

[1000]	valid_set's rmse: 473.777


 36%|███▌      | 9/25 [00:32<00:57,  3.62s/it]

[1000]	valid_set's rmse: 474.906


 40%|████      | 10/25 [00:41<01:16,  5.10s/it]

...

 72%|███████▏  | 18/25 [01:31<00:40,  5.75s/it]

[1000]	valid_set's rmse: 473.414


100%|██████████| 25/25 [02:06<00:00,  5.07s/it]
Fitted model: LightGBM/T1 ...
	-517.6578	 = Validation score   (-root_mean_squared_error)
	0.76s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: LightGBM/T2 ...
	-474.5714	 = Validation score   (-root_mean_squared_error)
	4.26s	 = Training   runtime
	0.02s	 = Validation runtime
Fitted model: LightGBM/T3 ...
	-477.1751	 = Validation score   (-root_mean_squared_error)
	2.91s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: LightGBM/T4 ...
	-476.102	 = Validation score   (-root_mean_squared_error)
	6.19s	 = Training   runtime
	0.02s	 = Validation runtime
Fitted model: LightGBM/T5 ...
	-475.4445	 = Validation score   (-root_mean_squared_error)
	0.88s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: LightGBM/T6 ...
	-476.0031	 = Validation score   (-root_mean_squared_error)
	4.82s	 = Training   runtime
	0.02s	 = Validation runtime
Fitted model: LightGBM/T7 ...
	-473.7189	 = Validation score   (-

Saved diagnostics to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_7_autogluon_categorical_clean_predictions.csv
Saved 50,000 competition predictions to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_7_autogluon_categorical_clean_submission.csv


,Id,Predicted
0,0,2733.381836
1,1,3624.259277
2,2,3196.371582
3,3,3165.918213
4,4,3300.301270
